# 04 — Backtesting & Evaluation

Feeds model predictions into Vectorbt with realistic transaction costs.
Computes Sharpe, Sortino, MDD, Profit Factor, Z-score for statistical significance.

**Anti-overfit guards:**
- Min 20 trades assertion raises ValueError
- Z-score < 1.96 raises ValueError (not statistically significant)
- Sharpe > 2.5 triggers warning (possible overfit)
- Win rate > 85% triggers warning (possible overfit)
- Asymmetric slippage (entry slippage > exit)

**Requires:** features.parquet + predictions.parquet + optimal_threshold.npy from Notebook 03.

In [ ]:
!pip install vectorbt matplotlib seaborn --quiet

In [ ]:
import numpy as np
import pandas as pd
import vectorbt as vbt
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_style("darkgrid")

## Configuration

In [ ]:
INITIAL_CAPITAL = 10000.0
POSITION_SIZE_PCT = 0.02
SPREAD_PIPS = 1.5
SLIPPAGE_PIPS = 0.5
STRESS_SPREAD_MULT = 3.0
MIN_TRADES = 20
Z_THRESHOLD = 1.96
MAX_ACCEPTABLE_SHARPE = 2.5
MAX_ACCEPTABLE_WIN_RATE = 0.85

CURRENCY_PAIRS = [
    "EURUSD", "GBPUSD", "USDJPY", "USDCAD", "AUDUSD",
    "NZDUSD", "USDCHF", "EURGBP", "EURJPY", "EURCHF",
]

print("Configuration loaded")

## Helper functions (inlined to avoid ml/ package dependency)

In [ ]:
def check_min_trades(num_trades):
    if num_trades < MIN_TRADES:
        raise ValueError(f"Only {num_trades} trades, need minimum {MIN_TRADES}")


def z_score(wins, losses):
    n = wins + losses
    if n < 2:
        return 0.0
    expected = n * 0.5
    std = np.sqrt(n * 0.5 * 0.5)
    return (wins - expected) / std if std > 0 else 0.0


def check_z(wins, losses):
    z = z_score(wins, losses)
    if z < Z_THRESHOLD:
        raise ValueError(f"Z-score {z:.3f} < {Z_THRESHOLD} — not statistically significant")


def check_overfit_warnings(sharpe, win_rate):
    if sharpe > MAX_ACCEPTABLE_SHARPE:
        warnings.warn(f"Sharpe {sharpe:.2f} > {MAX_ACCEPTABLE_SHARPE} — possible overfit")
    if win_rate > MAX_ACCEPTABLE_WIN_RATE:
        warnings.warn(f"Win rate {win_rate:.1%} > {MAX_ACCEPTABLE_WIN_RATE:.0%} — possible overfit")


def run_backtest(price, signals, spread=SPREAD_PIPS, slippage=SLIPPAGE_PIPS, capital=INITIAL_CAPITAL, size_pct=POSITION_SIZE_PCT, stress=False):
    if stress:
        spread *= STRESS_SPREAD_MULT
    portfolio = vbt.Portfolio.from_signals(
        price,
        entries=signals == 1,
        short_entries=signals == -1,
        direction="both",
        init_cash=capital,
        size=1.0,
        size_type="amount",
        fees=spread / 10000,
        slippage=slippage / 10000,
    )
    return portfolio


def sharpe_ratio(returns, rf=0.0, periods=252):
    excess = returns - rf / periods
    std = excess.std()
    return np.sqrt(periods) * excess.mean() / std if std > 0 and not np.isnan(std) else 0.0


def sortino_ratio(returns, rf=0.0, periods=252):
    excess = returns - rf / periods
    downside = excess[excess < 0]
    d_std = np.sqrt(np.mean(downside ** 2)) if len(downside) > 0 else 1e-8
    return np.sqrt(periods) * excess.mean() / d_std


def max_drawdown(equity):
    roll_max = equity.cummax()
    dd = (equity - roll_max) / roll_max
    return dd.min()


def profit_factor(gross_profit, gross_loss):
    return gross_profit / abs(gross_loss) if gross_loss != 0 else float("inf")


def compute_metrics(portfolio):
    stats = portfolio.stats()
    num_trades = len(portfolio.trades) if hasattr(portfolio, "trades") else 0
    win_rate = stats.get("Win Rate [%]", 0) / 100
    sharpe = stats.get("Sharpe Ratio", 0)

    wins = int(num_trades * win_rate)
    losses = num_trades - wins

    check_min_trades(num_trades)
    check_z(wins, losses)
    check_overfit_warnings(sharpe, win_rate)

    returns = portfolio.returns()
    equity = portfolio.value()

    return {
        "total_return_pct": stats.get("Total Return [%]", 0),
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino_ratio(returns),
        "max_drawdown_pct": stats.get("Max Drawdown [%]", 0),
        "num_trades": num_trades,
        "win_rate": win_rate,
        "profit_factor": stats.get("Profit Factor", 0),
        "z_score": z_score(wins, losses),
        "final_equity": equity.iloc[-1] if len(equity) > 0 else 0,
    }


print("Helper functions defined")

## Load data

In [ ]:
df = pd.read_parquet("/kaggle/input/forex-ml-02-feature-engineering/features.parquet")
print(f"Loaded features: {len(df)} rows")

pred_df = pd.read_parquet("/kaggle/input/forex-ml-03-model-training/predictions.parquet")
print(f"Loaded predictions: {len(pred_df)} rows")

threshold = float(np.load("/kaggle/input/forex-ml-03-model-training/optimal_threshold.npy"))
print(f"Loaded optimal threshold: {threshold:.2f}")

## Merge predictions with features and generate signals

In [ ]:
# Handle both old (single model) and new (multi-model) prediction formats
MODEL_VARIANTS = ["xgb"]
if "ensemble_probability" in pred_df.columns:
    MODEL_VARIANTS.append("ensemble")

idx_name = df.index.name or "index"
pred_idx_name = pred_df.index.name or "index"
df_reset = df.reset_index().rename(columns={idx_name: "date_idx"})
pred_reset = pred_df.reset_index().rename(columns={pred_idx_name: "date_idx"})
df = df_reset.merge(pred_reset, on=["date_idx", "pair"], how="inner")
df = df.set_index("date_idx").rename_axis(None)
df = df.dropna(subset=["close"])
print(f"Merged data: {len(df)} rows, columns: {df.columns.tolist()}")

for variant in MODEL_VARIANTS:
    prob_col = f"{variant}_probability"
    signal_col = f"{variant}_signal"
    if prob_col in df.columns:
        df[signal_col] = 0
        df.loc[df[prob_col] >= threshold, signal_col] = 1
        df.loc[df[prob_col] < (1 - threshold), signal_col] = -1
        counts = df[signal_col].value_counts()
        print(f"{variant}: Long={counts.get(1, 0)}, Short={counts.get(-1, 0)}, Flat={counts.get(0, 0)}")


## Per-pair backtests (Vectorbt requires separate backtest per asset)

In [ ]:
# Diagnostic: print per-pair signal distribution
print('\nPer-pair signal diagnostics:')
for pair in CURRENCY_PAIRS:
    sub = df[df["pair"] == pair]
    for variant in MODEL_VARIANTS:
        sc = f"{variant}_signal"
        if sc in sub.columns:
            vc = sub[sc].value_counts()
            chg = (sub[sc].diff() != 0).sum()
            print(f'  {pair} [{variant}]: {vc.to_dict()}, changes={chg}')
print()

pair_metrics = []
pair_portfolios = {}
for pair in CURRENCY_PAIRS:
    sub = df[df["pair"] == pair].copy()
    if len(sub) < 50:
        print(f"{pair}: skipping (only {len(sub)} rows)")
        continue
    for variant in MODEL_VARIANTS:
        signal_col = f"{variant}_signal"
        if signal_col not in sub.columns:
            continue
        try:
            p = run_backtest(sub["close"], sub[signal_col])
            m = compute_metrics(p)
            m["pair"] = pair
            m["model"] = variant
            pair_metrics.append(m)
            key = f"{pair}_{variant}"
            pair_portfolios[key] = p
            print(f"{pair} [{variant}]: Sharpe={m["sharpe_ratio"]:.2f}, Trades={m["num_trades"]}, Win={m["win_rate"]:.1%}")
        except (ValueError, Exception) as e:
            print(f"{pair} [{variant}]: FAILED — {e}")

pair_df = pd.DataFrame(pair_metrics)
print(f"\nPer-model per-pair results: {len(pair_df)} rows")
if not pair_df.empty:
    print(pair_df.groupby("model")[["sharpe_ratio", "num_trades", "win_rate"]].mean())


## Portfolio equity curve (aggregated from per-pair)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Build portfolio equity by summing per-pair equity curves
all_equities = []
for pair, p in pair_portfolios.items():
    eq = p.value() - INITIAL_CAPITAL
    eq.name = pair
    all_equities.append(eq)

if all_equities:
    combined = pd.concat(all_equities, axis=1).fillna(0).sum(axis=1) + INITIAL_CAPITAL
    axes[0].plot(combined.index, combined, label="Portfolio", linewidth=1)
    axes[0].set_title("Portfolio Equity Curve (Sum of Per-Pair PnL)")
    axes[0].set_ylabel("Portfolio Value ($)")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    pair_colors = plt.cm.tab10(np.linspace(0, 1, len(pair_portfolios)))
    for i, (pair, p) in enumerate(pair_portfolios.items()):
        eq = p.value()
        axes[1].plot(eq.index, eq, label=pair, linewidth=0.8, alpha=0.8, color=pair_colors[i])
    axes[1].set_title("Per-Pair Equity Curves")
    axes[1].set_ylabel("Portfolio Value ($)")
    axes[1].set_xlabel("Date")
    axes[1].legend(loc="best", fontsize=8)
    axes[1].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, "No equity data", ha="center", va="center", transform=axes[0].transAxes)
    axes[1].text(0.5, 0.5, "No equity data", ha="center", va="center", transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig("/kaggle/working/equity_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Drawdown plot

In [ ]:
if all_equities:
    cummax = combined.cummax()
    drawdown = (combined - cummax) / cummax

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.fill_between(drawdown.index, drawdown * 100, 0, color="red", alpha=0.3, label="Drawdown")
    ax.set_title("Portfolio Drawdown")
    ax.set_ylabel("Drawdown (%)")
    ax.set_xlabel("Date")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("/kaggle/working/drawdown.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No equity data for drawdown plot")

## Per-pair Sharpe comparison

In [ ]:
if len(pair_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ["green" if s > 0 else "red" for s in pair_df["sharpe_ratio"]]
    bars = ax.bar(pair_df["pair"], pair_df["sharpe_ratio"], color=colors, alpha=0.7)
    ax.axhline(y=0, color="black", linewidth=0.5)
    ax.axhline(y=1.0, color="green", linestyle="--", alpha=0.5, label="Good Sharpe")
    ax.axhline(y=2.0, color="orange", linestyle="--", alpha=0.5, label="Great Sharpe")
    ax.set_title("Per-Pair Sharpe Ratio")
    ax.set_ylabel("Sharpe Ratio")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("/kaggle/working/per_pair_sharpe.png", dpi=150, bbox_inches="tight")
    plt.show()

## Save results

In [ ]:
output_dir = Path("/kaggle/working")

if len(pair_df) > 0:
    pair_df.to_csv(output_dir / "pair_metrics.csv", index=False)
    print(f"Saved per-pair metrics: {output_dir / 'pair_metrics.csv'}")

    # Aggregate portfolio-level metrics
    mean_sharpe = pair_df["sharpe_ratio"].mean()
    mean_sortino = pair_df["sortino_ratio"].mean()
    total_trades = pair_df["num_trades"].sum()
    avg_win_rate = pair_df["win_rate"].mean()
    print(f"\n{'='*50}")
    print("PORTFOLIO AGGREGATE METRICS (mean of per-pair)")
    print(f"{'='*50}")
    print(f"  Mean Sharpe         : {mean_sharpe:.4f}")
    print(f"  Mean Sortino        : {mean_sortino:.4f}")
    print(f"  Total trades        : {total_trades}")
    print(f"  Avg win rate        : {avg_win_rate:.2%}")

    if all_equities:
        combined.to_frame("equity").to_parquet(output_dir / "portfolio_equity.parquet")
        print(f"Saved combined equity: {output_dir / 'portfolio_equity.parquet'}")

print("\nReady for notebook 05 — Daily Signal Generation")